# Data Collection & Preprocessing
## All Banks: CBE · BOA · Dashen

This notebook scrapes Google Play Store reviews for all three Ethiopian banks,
cleans the raw data, and saves a single structured CSV file.


In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.scraper import scrape_reviews
from src.preprocess import clean_reviews, missing_value_report

## 1. Configuration

All three banks are defined in one list. To add a new bank later,
just add one dictionary to the list — nothing else changes.

In [2]:
BANKS = [
    {"bank_name": "CBE",    "app_id": "com.combanketh.mobilebanking"},
    {"bank_name": "BOA",    "app_id": "com.boa.boaMobileBanking"},
    {"bank_name": "Dashen", "app_id": "com.dashen.dashensuperapp"},
]

LANG       = "en"
COUNTRY    = "et"
COUNT      = 500        # request more than 400 per bank as a buffer
OUTPUT_DIR = "../data/raw"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Scraping {len(BANKS)} banks — {[b['bank_name'] for b in BANKS]}")

Scraping 3 banks — ['CBE', 'BOA', 'Dashen']


## 2. Scrape All Banks

Loop through each bank, scrape its reviews, and collect the results
into a single combined raw DataFrame.

In [3]:
all_raw = []

for bank in BANKS:
    df = scrape_reviews(
        app_id    = bank["app_id"],
        bank_name = bank["bank_name"],
        lang      = LANG,
        country   = COUNTRY,
        count     = COUNT,
    )
    all_raw.append(df)

df_raw = pd.concat(all_raw, ignore_index=True)

print(f"\nTotal raw reviews collected: {len(df_raw)}")
print(f"Reviews per bank:")
print(df_raw["bank"].value_counts().to_string())
df_raw.head()

[CBE] Scraping 500 reviews for app: com.combanketh.mobilebanking
[CBE] Fetched 500 reviews.
[BOA] Scraping 500 reviews for app: com.boa.boaMobileBanking
[BOA] Fetched 500 reviews.
[Dashen] Scraping 500 reviews for app: com.dashen.dashensuperapp
[Dashen] Fetched 500 reviews.

Total raw reviews collected: 1500
Reviews per bank:
bank
CBE       500
BOA       500
Dashen    500


,review,rating,date,bank,source
0,Please make the CBE Noor toggle to be optional...,2,2026-05-17 09:25:49,CBE,Google Play
1,Amazing App,5,2026-05-17 04:11:07,CBE,Google Play
2,It stopped working on its own. When you check ...,1,2026-05-16 23:44:24,CBE,Google Play
3,The most backward and unstable financial app i...,1,2026-05-16 22:47:39,CBE,Google Play
4,ok,5,2026-05-16 21:47:39,CBE,Google Play


### Save raw data

Save the combined unmodified data before any cleaning —
so you can always re-run preprocessing without re-scraping.

In [4]:
raw_path = os.path.join(OUTPUT_DIR, "reviews_raw.csv")
df_raw.to_csv(raw_path, index=False)
print(f"Raw data saved → {raw_path}  ({len(df_raw)} rows)")

Raw data saved → ../data/raw\reviews_raw.csv  (1500 rows)


## 3. Explore Raw Data

Check for missing values, duplicates, and rating distributions
across all three banks before cleaning.

In [5]:
print("=== Shape ===")
print(df_raw.shape)

print("\n=== Data types ===")
print(df_raw.dtypes)

print("\n=== Missing values ===")
print(df_raw.isnull().sum())

print("\n=== Duplicates (same review text + same bank) ===")
print(df_raw.duplicated(subset=["review", "bank"]).sum())

print("\n=== Rating distribution per bank ===")
print(df_raw.groupby("bank")["rating"].value_counts().unstack(fill_value=0))

=== Shape ===
(1500, 5)

=== Data types ===
review               str
rating             int64
date      datetime64[us]
bank                 str
source               str
dtype: object

=== Missing values ===
review    0
rating    0
date      0
bank      0
source    0
dtype: int64

=== Duplicates (same review text + same bank) ===
291

=== Rating distribution per bank ===
rating    1   2   3   4    5
bank                        
BOA     149  16  18  37  280
CBE      75  13  33  45  334
Dashen   98  19  27  33  323


## 4. Preprocess & Clean

`clean_reviews()` from `src/preprocess.py` runs the same cleaning
pipeline on all banks at once:

1. Remove duplicate reviews (same text + same bank)
2. Drop rows missing review text or rating
3. Validate ratings are in the 1–5 range
4. Normalize dates to YYYY-MM-DD
5. Strip whitespace from review text

In [6]:
df_clean = clean_reviews(df_raw)

print(f"\nRows before : {len(df_raw)}")
print(f"Rows after  : {len(df_clean)}")
print(f"Rows removed: {len(df_raw) - len(df_clean)}")
df_clean.head()

[clean] Duplicates removed  : 291
[clean] Missing values dropped: 0
[clean] Invalid ratings removed: 0
[clean] Unparseable dates removed: 0
[clean] Final row count    : 1209

Rows before : 1500
Rows after  : 1209
Rows removed: 291


,review,rating,date,bank,source
0,Please make the CBE Noor toggle to be optional...,2,2026-05-17,CBE,Google Play
1,Amazing App,5,2026-05-17,CBE,Google Play
2,It stopped working on its own. When you check ...,1,2026-05-16,CBE,Google Play
3,The most backward and unstable financial app i...,1,2026-05-16,CBE,Google Play
4,ok,5,2026-05-16,CBE,Google Play


## 5. Data Quality Report

Verify the cleaned dataset meets the KPIs for Task 1.

In [7]:
total = len(df_clean)

print("═══ Overall ═══")
print(f"Total reviews : {total}")
print(f"KPI target    : 1,200 minimum")
print(f"KPI met       : {'YES ✓' if total >= 1200 else 'NO — expand date range or increase COUNT'}")

print("\n═══ Per bank ═══")
per_bank = df_clean.groupby("bank").agg(
    reviews   = ("review", "count"),
    avg_rating= ("rating", "mean"),
    date_from = ("date",   "min"),
    date_to   = ("date",   "max"),
).round(2)
print(per_bank.to_string())

print("\n═══ Missing values ═══")
print(missing_value_report(df_clean).to_string())

═══ Overall ═══
Total reviews : 1209
KPI target    : 1,200 minimum
KPI met       : YES ✓

═══ Per bank ═══
        reviews  avg_rating   date_from     date_to
bank                                               
BOA         410        3.33  2025-02-23  2026-05-16
CBE         376        3.89  2026-03-04  2026-05-17
Dashen      423        3.78  2025-08-14  2026-05-16

═══ Missing values ═══
        missing_count  missing_pct
review              0          0.0
rating              0          0.0
date                0          0.0
bank                0          0.0
source              0          0.0


## 6. Save Cleaned Dataset

In [8]:
clean_path = os.path.join(OUTPUT_DIR, "reviews_clean.csv")
df_clean.to_csv(clean_path, index=False)

print(f"Clean data saved → {clean_path}")
print(f"Columns          : {list(df_clean.columns)}")
print(f"Final shape      : {df_clean.shape}")

Clean data saved → ../data/raw\reviews_clean.csv
Columns          : ['review', 'rating', 'date', 'bank', 'source']
Final shape      : (1209, 5)


## 7. Final Preview

Sample 2 reviews from each bank to visually confirm the output looks correct.

In [9]:
df_clean.groupby("bank").apply(lambda x: x.sample(2, random_state=42)).reset_index(drop=True)

,review,rating,date,source
0,so good,5,2026-01-06,Google Play
1,profesionl,5,2026-03-02,Google Play
2,good application,5,2026-03-25,Google Play
3,ጥሩ ነው,5,2026-03-08,Google Play
4,my dashen is really proud,5,2026-01-24,Google Play
5,I just updated my app to the latest version an...,5,2025-11-01,Google Play
